# ECQL: LoRA против базовой модели

Перевод вопроса на русском в запрос на внутреннем языке ECQL.

Ноутбук работает в двух местах:
- **Google Colab, T4** - база грузится в 4 битах через bitsandbytes;
- **Mac, Apple Silicon** - bitsandbytes не работает, база грузится в bf16.

Порядок: 
- самопроверка метрик, 
- два прогона базовой модели, 
- обучение адаптера,
- прогон с адаптером, 
- сравнение.

## 1. Окружение

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q \
        transformers==5.15.1 \
        peft==0.20.0 \
        trl==1.10.0 \
        accelerate==1.14.0 \
        datasets==5.0.1 \
        nltk==3.10.3 \
        bitsandbytes

from importlib.metadata import PackageNotFoundError, version

for package in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "nltk", "bitsandbytes"):
    try:
        print(f"{package:16} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:16} не установлен")

torch            2.13.0
transformers     5.15.1
peft             0.20.0
trl              1.10.0
accelerate       1.14.0
datasets         5.0.1
nltk             3.10.3
bitsandbytes     не установлен


## 2. Код и данные

Код лежит в репозитории: `src/ecql_dataset`. 

В Colab репозиторий клонируется, локально берётся из каталога, где лежит ноутбук.

In [2]:
from pathlib import Path

REPO_URL = "https://github.com/samtakoy/llm-engineer-ecql-and-metrcis.git"

if IN_COLAB:
    ROOT = Path("/content") / Path(REPO_URL).stem
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))

DATASET = ROOT / "dataset" / "ecql"
print("корень:", ROOT)
print("датасет:", DATASET)

корень: /Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-21-training-metrics
датасет: /Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-21-training-metrics/dataset/ecql


In [3]:
import json

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]

train = read_jsonl(DATASET / "train.jsonl")
val = read_jsonl(DATASET / "val.jsonl")
test = read_jsonl(DATASET / "test.jsonl")


print(f"train {len(train)}, val {len(val)}, test {len(test)}, "
      f"из них challenge {sum(r['meta']['challenge'] for r in test)}")

print()
print(test[0]["input"])
print(test[0]["output"])

train 176, val 32, test 68, из них challenge 15

Отзывы из Пятигорска и Лермонтова
FETCH [REVIEWS] WHERE @city IS 'Пятигорск' || @city IS 'Лермонтов'


## 3. Устройство

Отсюда берутся размер батча и способ загрузки модели.

In [4]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    LOAD_IN_4BIT = True
    BATCH_SIZE = 4
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.bfloat16
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    LOAD_IN_4BIT = False
    BATCH_SIZE = 1

print({"устройство": DEVICE, "тип": str(DTYPE), "4 бита": LOAD_IN_4BIT, "батч": BATCH_SIZE})

{'устройство': 'mps', 'тип': 'torch.bfloat16', '4 бита': False, 'батч': 1}


## 4. Самопроверка метрик

Метрике подаются эталоны вместо ответов модели. Идеальный прогон обязан дать единицу по синтаксису, логике и строке и ноль галлюцинаций.

Проверка идёт до запуска модели: сломанная метрика делает бессмысленными все последующие цифры.

In [5]:
from ecql_dataset.notebook.eval.product import judge, self_check

print(self_check(records=test))

{'синтаксис': 1.0, 'логика': 1.0, 'сущность': 1.0, 'поля': 1.0, 'операторы': 1.0, 'значения': 1.0, 'суффикс': 1.0, 'галлюцинации': 0.0, 'строка': 1.0, 'ответов': 68}


- ответов 68 — весь тест;
- синтаксис 1.0 — все 68 разобрались по грамматике;
- логика 1.0 и пять частей по 1.0 — каждый совпал с собой;
- строка 1.0 — посимвольно тоже;
- галлюцинации 0.0 — чужого языка и выдуманных полей нет.

### Проверка на испорченных ответах

Самопроверка показала, что метрика не занижает. Теперь посмотрим — что не завышает.

Шесть ответов, каждый сломан по-своему. Логика должна упасть везде, кроме перестановки условий: там порядок другой, а смысл тот же.

In [6]:
reference = "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700 AS LIST"
cases = {
    "переставлены условия": "FETCH [PLACES] WHERE @price_rub BELOW 700 && @category IS 'food' AS LIST",
    "испорчено значение": "FETCH [PLACES] WHERE @category IS 'culture' && @price_rub BELOW 700 AS LIST",
    "перепутан оператор": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub ABOVE 700 AS LIST",
    "забыт суффикс": "FETCH [PLACES] WHERE @category IS 'food' && @price_rub BELOW 700",
    "SQL вместо ECQL": "SELECT * FROM places WHERE category = 'food'",
    "выдуманное поле": "FETCH [PLACES] WHERE @cuisine IS 'food' && @price_rub BELOW 700 AS LIST",
}

for name, prediction in cases.items():
    verdict = judge(prediction=prediction, reference=reference)
    print(f"{name:22} строка={int(verdict.exact)} логика={int(verdict.logic)} "
          f"галлюцинации={int(verdict.hallucination)}  {verdict.reason}")

переставлены условия   строка=0 логика=1 галлюцинации=0  
испорчено значение     строка=0 логика=0 галлюцинации=0  не совпало: значения
перепутан оператор     строка=0 логика=0 галлюцинации=0  не совпало: операторы, значения
забыт суффикс          строка=0 логика=0 галлюцинации=0  не совпало: суффикс
SQL вместо ECQL        строка=0 логика=0 галлюцинации=1  чужой язык запросов: запрос не разбирается по грамматике: "SELECT * FROM places WHERE category = 'food'"
выдуманное поле        строка=0 логика=0 галлюцинации=1  поля нет у [PLACES]: @cuisine


**Проверка: Текстовая метрика не заменяет логическую**

В каждом эталоне портится одно значение. Запрос остаётся правильным по
синтаксису и почти совпадает с эталоном как текст, но возвращает не те данные.

In [7]:
import re

from ecql_dataset.notebook.eval import text as text_metrics
from ecql_dataset.notebook.eval.product import evaluate

references = [record["output"] for record in test]
broken = [re.sub(r"'[^']*'", "'сломано'", reference, count=1) for reference in references]

rows = {}
for name, predictions in (("эталоны", references), ("одно значение испорчено", broken)):
    product, _ = evaluate(records=test, predictions=predictions)
    rows[name] = text_metrics.score(predictions=predictions, references=references) | {
        "логика": product["логика"],
        "синтаксис": product["синтаксис"],
    }

names = ["exact", "token_f1", "bleu", "meteor", "rouge_l", "cider", "синтаксис", "логика"]
print(f"{'метрика':12} {'эталоны':>10} {'испорчено':>12}")
for name in names:
    print(f"{name:12} {rows['эталоны'][name]:>10.3f} {rows['одно значение испорчено'][name]:>12.3f}")

метрика         эталоны    испорчено
exact             1.000        0.000
token_f1          1.000        0.943
bleu              1.000        0.850
meteor            1.000        0.940
rouge_l           1.000        0.943
cider             1.000        0.617
синтаксис         1.000        1.000
логика            1.000        0.000


Испорченный запрос синтаксически правильный и текстово почти совпадает с эталоном, но возвращает другие данные.

Все текстовые метрики, кроме exact, показывают 0.6–0.94 — «почти правильно». Логика показывает 0.

exact строгая, но слепая к перестановке условий.

Значит ориентироваться на них нельзя — подмену значения они не видят. 

Текстовые метрики отвечают «стало ли похоже на язык», логическая — «правильный ли запрос».

## 5. Промпты

Два промпта. 

Короткий — роль и схема данных, тот же текст лежит в поле `instruction` датасета. 

Полный — он же плюс правила языка словами - для базовой модели без дообучения.

Оба собираются из словаря и схемы модулем `ecql_dataset.prompt`, руками не пишутся: пополнится датасет — поменяются сами.

**Посмотрим на полный промпт:**

In [8]:
import json

from ecql_dataset.prompt import build_instruction

vocabulary = json.loads((DATASET / "source" / "vocabulary.json").read_text(encoding="utf-8"))

SHORT_PROMPT = build_instruction(vocabulary=vocabulary, with_rules=False)
FULL_PROMPT = build_instruction(vocabulary=vocabulary, with_rules=True)

assert SHORT_PROMPT == train[0]["instruction"], "короткий промпт разошёлся с датасетом"
assert FULL_PROMPT.startswith(SHORT_PROMPT), "полный промпт обязан начинаться коротким"

print(f"короткий {len(SHORT_PROMPT)} знаков, полный {len(FULL_PROMPT)}")
print()
print("Полный промпт для базовой модели:\n---")
print(FULL_PROMPT)


короткий 2018 знаков, полный 3508

Полный промпт для базовой модели:
---
Ты переводишь вопрос человека в запрос на ECQL - внутреннем языке запросов компании.
Отвечай одной строкой запроса, без пояснений.

Схема данных:

[PLACES]
- @name, строка, примеры: Фитнесс центр, Дача "Золотой курган" Л.К.Менякова, Кофейня Dr.Coff, Интурист и т.д.
- @city, строка, примеры: Москва, Волгоград, Пятигорск и т.д.; объект вне населённого пункта - 'вне городов'
- @category, строка, допустимые значения: culture, food, shopping, lodging, nature, service, water, activity, transport
- @price_rub, число, примеры: 300, 400, 500, 1300 и т.д.
- @price_kind, строка, допустимые значения: average_check, per_night
- @object_kind, строка, допустимые значения: monument, ensemble, heritage_site
- @heritage_status, строка, допустимые значения: regional, federal, local
- @wheelchair, строка, допустимые значения: yes, no, limited

[REVIEWS]
- @name, строка, примеры: Интурист, Пространство лофт, Кристелла, Олимпия и т.д.


### Обучающий пример целиком

In [9]:
from ecql_dataset.prompt import build_training_messages

example = train[0]
messages = build_training_messages(
    instruction=SHORT_PROMPT,
    question=example["input"],
    answer=example["output"],
)

for message in messages:
    print(f"--- {message['role']} ---")
    print(message["content"])

--- system ---
Ты переводишь вопрос человека в запрос на ECQL - внутреннем языке запросов компании.
Отвечай одной строкой запроса, без пояснений.

Схема данных:

[PLACES]
- @name, строка, примеры: Фитнесс центр, Дача "Золотой курган" Л.К.Менякова, Кофейня Dr.Coff, Интурист и т.д.
- @city, строка, примеры: Москва, Волгоград, Пятигорск и т.д.; объект вне населённого пункта - 'вне городов'
- @category, строка, допустимые значения: culture, food, shopping, lodging, nature, service, water, activity, transport
- @price_rub, число, примеры: 300, 400, 500, 1300 и т.д.
- @price_kind, строка, допустимые значения: average_check, per_night
- @object_kind, строка, допустимые значения: monument, ensemble, heritage_site
- @heritage_status, строка, допустимые значения: regional, federal, local
- @wheelchair, строка, допустимые значения: yes, no, limited

[REVIEWS]
- @name, строка, примеры: Интурист, Пространство лофт, Кристелла, Олимпия и т.д.
- @city, строка, примеры: Москва, Волгоград, Пятигорск и т

## 6. Модель

Отладка идёт на Qwen3-0.6B.

In [10]:
# TODO: сюда переедет load_model из ecql_dataset.notebook.generate
# Переедет: загрузка токенизатора (padding_side="left", pad_token из eos),
# ветка 4 бит через BitsAndBytesConfig для Colab, загрузка модели и model.eval().

from ecql_dataset.notebook.generate import load_model

MODEL_NAME = "Qwen/Qwen3-0.6B"

model, tokenizer = load_model(model_name=MODEL_NAME, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT)

parameters = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: параметров {parameters / 1e9:.2f} млрд, устройство {model.device}")

/Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-21-training-metrics/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 34112.52it/s]


Qwen/Qwen3-0.6B: параметров 0.60 млрд, устройство cpu


### Что уходит в модель

In [11]:
from ecql_dataset.notebook.generate import render_prompt

print(render_prompt(tokenizer=tokenizer, instruction="Инструкция сюда", question=test[0]["input"]))

<|im_start|>system
Инструкция сюда<|im_end|>
<|im_start|>user
Отзывы из Пятигорска и Лермонтова<|im_end|>
<|im_start|>assistant
<think>

</think>




### Генерация

Из ответа берётся строка запроса — `extract_query` отбрасывает обрамление кода, размышления и пояснения вокруг.

Проверка на трёх вопросах, до полного прогона.

In [12]:
from ecql_dataset.notebook.generate import generate

probe = test[:3]
answers = generate(
    model=model,
    tokenizer=tokenizer,
    questions=[record["input"] for record in probe],
    instruction=SHORT_PROMPT,
    batch_size=BATCH_SIZE,
)

for record, answer in zip(probe, answers):
    print(record["input"])
    print("эталон:", record["output"])
    print("ответ: ", answer)
    print()

Отзывы из Пятигорска и Лермонтова
эталон: FETCH [REVIEWS] WHERE @city IS 'Пятигорск' || @city IS 'Лермонтов'
ответ:  @city Пятигорск, @name Лермонтова.

Трамвай за двадцать-сорок рублей
эталон: FETCH [FARES] WHERE @transport IS 'tram' && @price_rub ABOVE 20 && @price_rub BELOW 40
ответ:  @transport, @route_start, @route_end, @fare_class, @price_rub

«Оазис» — что там с чистотой?
эталон: FETCH [REVIEWS] WHERE @name IS 'Оазис' && @aspects CONTAINS 'чистота'
ответ:  SELECT @heritage_status AS "чистота" WHERE @object_class = 'monument';

